# Simulation de la Ligue des champions

**Objectif :** estimer, pour chacun des 36 clubs, ses chances d'atteindre chaque étape de la compétition (top 8, huitièmes, quarts, ..., victoire finale).

**Méthode :** on simule la compétition entière (phase de ligue puis phase finale) des milliers de fois avec un modèle de buts aléatoire, puis on compte la fréquence de chaque résultat (méthode de Monte-Carlo).

## 1. Chargement du projet

In [ ]:
using Pkg
Pkg.activate("..")

using SimulationsProjet, DataFrames, GLMakie
GLMakie.activate!(inline = true) # graphiques dans le notebook

N = 10000 # nombre de simulations

## 2. Données

- `classement_club.csv` : coefficients UEFA (club et pays) et Elo des 36 clubs ;
- `matchs.csv` : calendrier des 8 journées de la phase de ligue ;
- `tirs_au_but.csv` : séances de tirs au but jouées et tirs réussis par club.

In [ ]:
first(CLASSEMENT, 10)

In [ ]:
first(CALENDRIER, 5)

In [ ]:
first(STATS_TAB, 5)

### Force d'un club

Chaque club a un poids $w$. Avec les coefficients UEFA :

$$F = 0.8 \times \text{Coeff}_\text{club} + 0.2 \times \text{Coeff}_\text{pays}, \qquad w = \frac{F - \overline{F}}{\max F - \min F}$$

Avec l'Elo : $w = \text{Elo} / 400$.

### Tirs au but

La probabilité de réussir un tir au but est estimée à partir des données du club, ramenée vers 75 % quand il y a peu de tirs :

$$p = \frac{\text{réussis} + 0.75 \times 10}{\text{tentés} + 10}$$

## 3. Modèles de buts

Pour un match entre $A$ (à domicile) et $B$, avec $\Delta = w_A - w_B$ :

$$\lambda_A = \mu \times 1.2 \times e^{\Delta}, \qquad \lambda_B = \mu \times e^{-\Delta}$$

($1.2$ = avantage du terrain, $\mu$ règle le nombre moyen de buts, environ 3 par match.)

| Modèle | Idée | Paramètres par défaut |
|---|---|---|
| `ModelePoisson` | buts indépendants $\sim \mathcal{P}(\lambda)$ | $\mu = 1.25$ |
| `ModelePoissonBivariee` | Poisson + une partie commune $X \sim \mathcal{P}(\lambda_c e^{-|\Delta|})$ aux deux équipes | $\mu = 1$, $\lambda_c = 0.4$ |
| `ModeleBinomialeNegative` | même moyenne que Poisson mais plus dispersé | $\mu = 1.25$, $p = 0.8$ |
| `ModeleDixonColes` | Poisson corrigé sur les petits scores (0-0, 1-0, 0-1, 1-1) | $\mu = 1.25$, $\rho = -0.1$ |

**Loi de Poisson :** $P(X = k) = \dfrac{\lambda^k e^{-\lambda}}{k!}$

**Dixon-Coles :** $P(X = a, Y = b) \propto \tau(a, b)\, P(X = a)\, P(Y = b)$, où $\tau$ modifie seulement les scores 0-0, 1-0, 0-1 et 1-1.

## 4. Déroulement d'une simulation

1. **Phase de ligue :** les 8 journées du calendrier, 3 points par victoire, 1 par nul.
2. **Classement :** les 8 premiers vont en huitièmes, les places 9 à 24 jouent les barrages, les 12 derniers sont éliminés.
3. **Phase finale :** matchs aller-retour (prolongation si égalité, puis tirs au but), finale sur un seul match.

## 5. Résultats avec Dixon-Coles (poids UEFA)

In [ ]:
df = classement_stats(rand(N, ModeleDixonColes(); seed = 1))
show(df, allrows = true)

In [ ]:
graphique_chances(df)

In [ ]:
graphique_chances(df; etape = :Quart)

**Analyse :** *(à compléter : quels sont les favoris ? les écarts sont-ils grands ? des surprises ?)*

## 6. Poids UEFA ou Elo ?

In [ ]:
df_elo = classement_stats(rand(N, ModeleDixonColes(), "Elo"; seed = 1))
graphique_chances(df_elo)

**Analyse :** *(à compléter : comparer avec le graphique de la partie 5. Quels clubs changent le plus ?)*

## 7. Comparaison des modèles

In [ ]:
comparaison_modeles(N)

**Analyse :** *(à compléter : les modèles donnent-ils les mêmes favoris ? lequel resserre le plus les chances ?)*

## 8. Effet de μ (nombre moyen de buts)

In [ ]:
for μ in [1.0, 1.25, 1.5]
    d = classement_stats(rand(N, ModeleDixonColes(μ = μ); seed = 1))
    println("μ = $μ : ", first(sort(d, :Vainqueur, rev = true), 3)[:, [:Club, :Vainqueur]])
end

**Analyse :** *(à compléter : avec plus de buts, le hasard compte-t-il moins ?)*

## 9. Nombre de simulations suffisant ?

In [ ]:
for n in [1000, 10000, 50000]
    d = classement_stats(rand(n, ModeleDixonColes(); seed = 2))
    println("n = $n : ", first(sort(d, :Vainqueur, rev = true), 3)[:, [:Club, :Vainqueur]])
end

Pour une probabilité $p$ estimée avec $n$ simulations, l'erreur type vaut $\sqrt{p(1-p)/n}$ : pour $p = 0.15$ et $n = 10000$, environ $0.36$ point de pourcentage.

## 10. Conclusion et limites

*(à compléter)*

Pistes :
- la force d'un club ne change pas pendant la compétition (blessures, forme) ;
- l'Elo pourrait être mis à jour après chaque match simulé ;
- le tirage de la phase finale est simplifié.